# Naive Bayes for Conspiracy Detection
## Predicting conspiracy labels and marker types

This notebook implements Naive Bayes classifier for:
1. **Conspiracy Label Prediction**: yes/no/cant_tell (multiclass)
2. **Marker Type Prediction**: Action, Actor, Effect, Evidence, Victim (binary classification for each)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


In [ ]:
# Load and merge all features
BASE = Path('../')
PROC = BASE / 'data_processed'
FEAT = BASE / 'features'

# Load base data
df = pd.read_parquet(PROC / 'data_processed.parquet')
if df is None or len(df) == 0:
    df = pd.read_csv(PROC / 'data_clean.csv')

print(f"Base data: {df.shape}")

# Load feature files
feature_files = [
    FEAT / 'lexical_complexity.csv',
    FEAT / 'discourse_markers.csv',
    FEAT / 'sentiment_emotion.csv',
    FEAT / 'token_pos_counts.csv',
    FEAT / 'readability_scores.csv',
    FEAT / 'ma_ttr_mtld_scores.csv',
    FEAT / 'pos_analysis.csv'
]

merged = df.copy()
for feat_file in feature_files:
    if feat_file.exists():
        try:
            feat_df = pd.read_csv(feat_file)
            if '_id' in merged.columns and '_id' in feat_df.columns:
                merged = merged.merge(feat_df, on='_id', how='left')
                print(f"Merged {feat_file.name}")
        except Exception as e:
            print(f"Could not load {feat_file.name}: {e}")

print(f"\nFinal merged shape: {merged.shape}")
print(f"Columns: {len(merged.columns)}")


In [ ]:
# Extract numeric features
numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
# Remove _id and other non-feature columns
numeric_cols = [c for c in numeric_cols if c not in ['_id']]
print(f"Number of numeric features: {len(numeric_cols)}")

X = merged[numeric_cols].fillna(0)
print(f"Feature matrix shape: {X.shape}")
print(f"Missing values: {X.isna().sum().sum()}")


## Task 1: Conspiracy Label Prediction (yes/no/cant_tell)


In [ ]:
# Task 1: Conspiracy Label Prediction (yes/no/cant_tell)
y_conspiracy = merged['conspiracy'].copy()
# Filter out any null labels
mask = y_conspiracy.notna()
X_clean = X[mask]
y_clean = y_conspiracy[mask]

print(f"Samples for conspiracy prediction: {len(y_clean)}")
print(f"Label distribution:")
print(y_clean.value_counts())


In [ ]:
# Encode labels
le_conspiracy = LabelEncoder()
y_encoded = le_conspiracy.fit_transform(y_clean)
print(f"Encoded labels: {dict(zip(le_conspiracy.classes_, range(len(le_conspiracy.classes_))))}")

# Create pipeline with scaling and Naive Bayes
# Use GaussianNB for continuous features
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

# 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']
cv_results = cross_validate(pipeline, X_clean, y_encoded, cv=cv, scoring=scoring, n_jobs=-1)

print("\n=== Conspiracy Label Prediction (5-fold CV) ===")
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Train final model
pipeline.fit(X_clean, y_encoded)
print("\nModel trained successfully!")


## Task 2: Marker Type Prediction (Action, Actor, Effect, Evidence, Victim)


In [ ]:
# Task 2: Marker Type Prediction (Multi-label)
MARKER_TYPES = ["Action", "Actor", "Effect", "Evidence", "Victim"]

# Extract marker presence for each type
def extract_markers(row):
    markers = row.get('markers', [])
    if pd.isna(markers) or markers == '':
        return {marker: 0 for marker in MARKER_TYPES}
    
    if isinstance(markers, str):
        try:
            markers = json.loads(markers)
        except:
            return {marker: 0 for marker in MARKER_TYPES}
    
    marker_dict = {marker: 0 for marker in MARKER_TYPES}
    if isinstance(markers, list):
        for m in markers:
            if isinstance(m, dict) and 'type' in m:
                marker_type = m['type']
                if marker_type in MARKER_TYPES:
                    marker_dict[marker_type] = 1
    
    return marker_dict

marker_data = merged.apply(extract_markers, axis=1)
marker_df = pd.DataFrame(list(marker_data))

print(f"Marker presence counts:")
print(marker_df.sum())
print(f"\nMarker co-occurrence:")
print(marker_df.corr())


In [ ]:
# Train separate binary classifiers for each marker type
marker_results = {}

for marker_type in MARKER_TYPES:
    y_marker = marker_df[marker_type].values
    
    # Use same X_clean (conspiracy-labeled data)
    pipeline_marker = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('nb', GaussianNB())
    ])
    
    cv_marker = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_validate(pipeline_marker, X_clean, y_marker, 
                             cv=cv_marker, scoring=['accuracy', 'f1', 'precision', 'recall'], 
                             n_jobs=-1)
    
    marker_results[marker_type] = {
        'accuracy': cv_scores['test_accuracy'].mean(),
        'f1': cv_scores['test_f1'].mean(),
        'precision': cv_scores['test_precision'].mean(),
        'recall': cv_scores['test_recall'].mean()
    }
    
    print(f"\n{marker_type}:")
    print(f"  Accuracy: {cv_scores['test_accuracy'].mean():.4f} (+/- {cv_scores['test_accuracy'].std() * 2:.4f})")
    print(f"  F1: {cv_scores['test_f1'].mean():.4f} (+/- {cv_scores['test_f1'].std() * 2:.4f})")
    print(f"  Precision: {cv_scores['test_precision'].mean():.4f} (+/- {cv_scores['test_precision'].std() * 2:.4f})")
    print(f"  Recall: {cv_scores['test_recall'].mean():.4f} (+/- {cv_scores['test_recall'].std() * 2:.4f})")
    
    # Train final model
    pipeline_marker.fit(X_clean, y_marker)

print("\n=== Marker Type Prediction Summary ===")
marker_summary = pd.DataFrame(marker_results).T
print(marker_summary)


## Testing on Dev Set


In [ ]:
# Load dev set
dev_file = Path('../../dev_rehydrated.jsonl')
dev_data = []
with open(dev_file, 'r', encoding='utf-8') as f:
    for line in f:
        dev_data.append(json.loads(line.strip()))

dev_df = pd.DataFrame(dev_data)
print(f"Dev set size: {len(dev_df)}")
print(f"Dev set columns: {dev_df.columns.tolist()}")

print("\nNote: Dev set features need to be computed using the EDA pipeline first.")
print("This notebook assumes dev set features will be merged similarly to training data.")
print("\nOnce dev features are available, use:")
print("  - pipeline.predict(X_dev) for conspiracy labels")
print("  - pipeline_marker.predict(X_dev) for each marker type")
